# Estimating and eliminating trend: lab notebook

**Statistical Modelling and Forecasting, Statistics background for forecasting**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, acf

### 1. The numerical problem: exact quadratic y = 10 + 2t + t^2

In [ ]:
# 1. The numerical problem: exact quadratic y = 10 + 2t + t^2
t = np.arange(1, 7)
y = 10 + 2 * t + t ** 2
print("y:", y.tolist())
print("first differences :", np.diff(y).tolist())
print("second differences:", np.diff(y, 2).tolist(), "<- constant = 2c")
b, a = np.polyfit(t, y, 1)
print("linear fit: slope", round(b, 3), "intercept", round(a, 3), "| forecast t=7:", round(a + b * 7, 3), "| residual at t=1:", round(y[0] - (a + b), 3))

### 2. Deterministic against stochastic trend

In [ ]:
# 2. Deterministic against stochastic trend
rng = np.random.default_rng(2)
n = 300
tt = np.arange(n)
det = 0.4 * tt + 3 * rng.normal(size=n)                       # trend-stationary
sto = np.cumsum(0.4 + 3 * rng.normal(size=n))                 # random walk with drift

def adf_stat(x):
    return round(adfuller(x, maxlag=1, regression="c", autolag=None)[0], 2)

def detrend(x):
    return x - np.polyval(np.polyfit(tt[:len(x)], x, 1), tt[:len(x)])

def adf_ct(x):
    return round(adfuller(x, maxlag=1, regression="ct", autolag=None)[0], 2)

print("ADF(c)  of the original series  : deterministic", adf_stat(det), "| stochastic", adf_stat(sto))
print("ADF(ct) of the original series  : deterministic", adf_ct(det), "| stochastic", adf_ct(sto))
print("after detrending (subtract line): deterministic", adf_stat(detrend(det)), "| stochastic", adf_stat(detrend(sto)))
print("after first difference          : deterministic", adf_stat(np.diff(det)), "| stochastic", adf_stat(np.diff(sto)))
print("critical values (5%): c about -2.87; ct about -3.41. After detrending use the stricter -3.41.")

### 3. Over-differencing a deterministic trend gives r1 near -0.5

In [ ]:
# 3. Over-differencing a deterministic trend gives r1 near -0.5
print("r1 after detrending:", round(acf(detrend(det), nlags=1)[1], 2), "| r1 after differencing:", round(acf(np.diff(det), nlags=1)[1], 2))

### 4. Real data: log GDP needs one difference, and the growth rate is stationary

In [ ]:
# 4. Real data: log GDP needs one difference, and the growth rate is stationary
g = np.log(sm.datasets.macrodata.load_pandas().data["realgdp"])
print("log GDP ADF(c):", adf_stat(g.values), "| growth rate ADF(c):", adf_stat(np.diff(g.values)))
print("mean quarterly growth: %.2f%%" % (100 * np.diff(g.values).mean()))

### 5. Aggregation: yearly mean of quarterly GDP shows the trend without seasonality

In [ ]:
# 5. Aggregation: yearly mean of quarterly GDP shows the trend without seasonality
df = sm.datasets.macrodata.load_pandas().data
yearly = df.groupby("year")["realgdp"].mean()
print(yearly.loc[[1960, 1980, 2000]].round(0).to_dict())

## Expected output

Running all cells above should print:

```
y: [13, 18, 25, 34, 45, 58]
first differences : [5, 7, 9, 11, 13]
second differences: [2, 2, 2, 2] <- constant = 2c
linear fit: slope 9.0 intercept 0.667 | forecast t=7: 63.667 | residual at t=1: 3.333
ADF(c)  of the original series  : deterministic -0.61 | stochastic -2.43
ADF(ct) of the original series  : deterministic -12.2 | stochastic -3.02
after detrending (subtract line): deterministic -12.22 | stochastic -3.02
after first difference          : deterministic -20.12 | stochastic -11.87
critical values (5%): c about -2.87; ct about -3.41. After detrending use the stricter -3.41.
r1 after detrending: -0.03 | r1 after differencing: -0.52
log GDP ADF(c): -1.82 | growth rate ADF(c): -6.97
mean quarterly growth: 0.78%
{1960.0: 2831.0, 1980.0: 5839.0, 2000.0: 11226.0}
```

## Practice exercises

Try these in the cells below.

1. Choose "Linear trend" and compare linear detrending with first differencing. Compare r₁ of the two results.
2. Choose "Stochastic trend (random walk with drift)" and detrend it with a polynomial. Does the ADF test accept the result as stationary?
3. Choose "Quadratic trend" and find the smallest number of differences that makes the series flat.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
